In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### 1. Installation and Import Libraries

In [2]:
!pip install yfinance -q

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf

### 2. Define the study period

In [4]:
start_date = "2020-01-01"
end_date = "2026-01-01"

### 3. Download S&P 500 and VIX

In [5]:
#^GSPC is S&P 500 Index and provies information about the overall direction of the US equity market.
#^VIX is CBOE volatility index (a real-time market indeex that measures expected **30-D stock market volatity**, known as "fear index")
sp500_raw = yf.download(
    "^GSPC",
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)

vix_raw = yf.download(
    "^VIX",
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)

print("S&P 500 shape:", sp500_raw.shape)
print("VIX shape:", vix_raw.shape)

display(sp500_raw.head())
display(vix_raw.head())

S&P 500 shape: (1508, 6)
VIX shape: (1508, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,,
2020-01-02,3257.850098,3257.850098,3258.139893,3235.530029,3244.669922,3459930000
2020-01-03,3234.850098,3234.850098,3246.149902,3222.340088,3226.360107,3484700000
2020-01-06,3246.280029,3246.280029,3246.840088,3214.639893,3217.550049,3702460000
2020-01-07,3237.179932,3237.179932,3244.909912,3232.429932,3241.860107,3435910000
2020-01-08,3253.050049,3253.050049,3267.070068,3236.669922,3238.590088,3726840000


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^VIX,^VIX,^VIX,^VIX,^VIX,^VIX
Date,,,,,,
2020-01-02,12.47,12.47,13.720000,12.42,13.46,0
2020-01-03,14.02,14.02,16.200001,13.13,15.01,0
2020-01-06,13.85,13.85,16.389999,13.54,15.45,0
2020-01-07,13.79,13.79,14.460000,13.39,13.84,0
2020-01-08,13.45,13.45,15.240000,12.83,15.16,0


In [6]:
# Extract the closing values from each dataset.

sp500_close = sp500_raw[["Close"]].copy()
sp500_close.columns = ["market_close"]

vix_close = vix_raw[["Close"]].copy()
vix_close.columns = ["vix_close"]

# Keep only dates available in both series.
market_context = sp500_close.join(
    vix_close,
    how="inner"
)

market_context = (
    market_context
    .reset_index()
    .rename(columns={"Date": "date"})
)

market_context["date"] = pd.to_datetime(
    market_context["date"],
    errors="coerce"
)

display(market_context.head())

,date,market_close,vix_close
0,2020-01-02,3257.850098,12.47
1,2020-01-03,3234.850098,14.02
2,2020-01-06,3246.280029,13.85
3,2020-01-07,3237.179932,13.79
4,2020-01-08,3253.050049,13.45


In [7]:
print("Missing values before feature creation:")
print(market_context.isna().sum())

print("\nDate range:")
print(
    market_context["date"].min(),
    "to",
    market_context["date"].max()
)

Missing values before feature creation:
date            0
market_close    0
vix_close       0
dtype: int64

Date range:
2020-01-02 00:00:00 to 2025-12-31 00:00:00


In [8]:
# Calculate same-day market changes for validation and later feature creation.

market_context["market_return"] = (
    market_context["market_close"]
    .pct_change()
)

market_context["vix_change"] = (
    market_context["vix_close"]
    .pct_change()
)

# Create leakage-safe features using information from the previous
# trading day.

market_context["market_return_lag_1"] = (
    market_context["market_return"]
    .shift(1)
)

market_context["vix_close_lag_1"] = (
    market_context["vix_close"]
    .shift(1)
)

market_context["vix_change_lag_1"] = (
    market_context["vix_change"]
    .shift(1)
)

In [9]:
print("Final missing values:")
print(market_context.isna().sum())

print("\nDuplicate dates:")
print(
    market_context.duplicated(
        subset=["date"]
    ).sum()
)

display(market_context.head())

Final missing values:
date                   0
market_close           0
vix_close              0
market_return          1
vix_change             1
market_return_lag_1    2
vix_close_lag_1        1
vix_change_lag_1       2
dtype: int64

Duplicate dates:
0


,date,market_close,vix_close,market_return,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,2020-01-02,3257.850098,12.47,NaN,NaN,NaN,NaN,NaN
1,2020-01-03,3234.850098,14.02,-0.007060,0.124298,NaN,12.47,NaN
2,2020-01-06,3246.280029,13.85,0.003533,-0.012126,-0.007060,14.02,0.124298
3,2020-01-07,3237.179932,13.79,-0.002803,-0.004332,0.003533,13.85,-0.012126
4,2020-01-08,3253.050049,13.45,0.004902,-0.024656,-0.002803,13.79,-0.004332


In [10]:
print("Shape:", market_context.shape)

print("\nColumns:")
print(market_context.columns.tolist())

print("\nDate range:")
print(market_context["date"].min(), "to", market_context["date"].max())

print("\nMissing values:")
print(market_context.isna().sum())

Shape: (1508, 8)

Columns:
['date', 'market_close', 'vix_close', 'market_return', 'vix_change', 'market_return_lag_1', 'vix_close_lag_1', 'vix_change_lag_1']

Date range:
2020-01-02 00:00:00 to 2025-12-31 00:00:00

Missing values:
date                   0
market_close           0
vix_close              0
market_return          1
vix_change             1
market_return_lag_1    2
vix_close_lag_1        1
vix_change_lag_1       2
dtype: int64


### 5. Create market Return and VIX change

In [11]:
print("Market return summary:")
display(market_context["market_return"].describe())

print("VIX close summary:")
display(market_context["vix_close"].describe())

print("VIX change summary:")
display(market_context["vix_change"].describe())

Market return summary:


count    1507.000000
mean        0.000580
std         0.013183
min        -0.119841
25%        -0.004835
50%         0.000969
75%         0.006871
max         0.095154
Name: market_return, dtype: float64

VIX close summary:


count    1508.000000
mean       21.005630
std         7.902060
min        11.860000
25%        15.947500
50%        19.070000
75%        24.110001
max        82.690002
Name: vix_close, dtype: float64

VIX change summary:


count    1507.000000
mean        0.003383
std         0.084498
min        -0.357539
25%        -0.041961
50%        -0.007902
75%         0.035374
max         0.740391
Name: vix_change, dtype: float64

In [12]:
market_context_processed = market_context[
    [
        "date",
        "market_close",
        "market_return",
        "vix_close",
        "vix_change",
        "market_return_lag_1",
        "vix_close_lag_1",
        "vix_change_lag_1"
    ]
].copy()

In [13]:
output_file = (
    "/kaggle/working/"
    "market_context_processed_2020_2025.csv"
)

market_context_processed.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Final shape:", market_context_processed.shape)

Saved: /kaggle/working/market_context_processed_2020_2025.csv
Final shape: (1508, 8)


In [14]:
market_check = pd.read_csv(output_file)

print("Saved file shape:", market_check.shape)
print("\nSaved file missing values:")
print(market_check.isna().sum())

display(market_check.head())

Saved file shape: (1508, 8)

Saved file missing values:
date                   0
market_close           0
market_return          1
vix_close              0
vix_change             1
market_return_lag_1    2
vix_close_lag_1        1
vix_change_lag_1       2
dtype: int64


,date,market_close,market_return,vix_close,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,2020-01-02,3257.850098,NaN,12.47,NaN,NaN,NaN,NaN
1,2020-01-03,3234.850098,-0.007060,14.02,0.124298,NaN,12.47,NaN
2,2020-01-06,3246.280029,0.003533,13.85,-0.012126,-0.007060,14.02,0.124298
3,2020-01-07,3237.179932,-0.002803,13.79,-0.004332,0.003533,13.85,-0.012126
4,2020-01-08,3253.050049,0.004902,13.45,-0.024656,-0.002803,13.79,-0.004332
